# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [1]:
import os

# Async CUDA allocator
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# If cuDNN autotune fails, fall back to a safe (but slower) algorithm.
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true' 

In [2]:
# Disable all auto-JIT clustering at the process level
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"

### 1.2. Imports

In [3]:
from _imports import * # Centralized file containing all imports
from spektral.layers import GraphMasking, GlobalAvgPool

2025-07-17 08:33:05.442094: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-17 08:33:05.457690: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752751985.474905   67479 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752751985.480159   67479 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-17 08:33:05.506111: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [4]:
# Disable all XLA auto-JIT compilation
tf.config.optimizer.set_jit(False)

# Spektral’s GCNConv uses a sparse-dense matmul under the hood.
# XLA’s GPU JIT compiler does not support that op.

### 1.3. GPU Management

In [5]:
# Specify GPU to use (e.g., GPU:0, CPU:-1)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
get_gpu_info()


TensorFlow GPU Monitor - 2025-07-17 08:33:07
TensorFlow Configuration
Version        : 2.18.0
CUDA Support   : Yes
CUDA Version   : 12.5.1
cuDNN Version  : 9

GPU Information
GPU Name                      Memory Usage         Temp   Util  
--------------------------------------------------------------------------------
0   NVIDIA GeForce RTX 3060      0.3GB /   12.0GB  44C    36%   
1   NVIDIA GeForce RTX 4070      0.2GB /   12.0GB  43C    2%    



2025-07-17 08:33:08.115649: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1752751988.115681   67479 gpu_process_state.cc:201] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1752751988.116633   67479 gpu_device.cc:2022] Created device /device:GPU:0 with 10174 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070, pci bus id: 0000:c3:00.0, compute capability: 8.9


## 2. Run Parameters 

In [6]:
NUM_TRIALS = 1000
EPOCHS = 100

DATA_SEED = 99
TRAIN_SEED = 111

In [7]:
# Number of top trials to save
TOP_K = 3

# Order to rank trials by:
# "ascending" -> the lowest value is the best
# "descending" -> the highest value is the best
ORDER = "descending"

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "test_accuracy_s009_full"

# Direction of optimization:
# "minimize" -> the lowest value is the best
# "maximize" -> the highest value is the best
DIRECTION = "minimize"

In [8]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

In [9]:
# Set to an existing dir to resume training
RUN_DIR = "runs/nas_gnn_v0.0"  # (e.g. "runs/nas_1")

## 3. Data Loading and Preprocessing

In [10]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()

/home/matheus/src/RayWise/src/_load_dataset.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)
/home/matheus/src/RayWise/src/_load_dataset.py:65: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_val = s008_y_val.astype(np.float32)


Shape before conversion: (9234, 8, 32)
Shape after conversion: (9234,)
y_train shape: (9234,)
coord_input shape: (9234, 2)
lidar_input shape: (9234, 20, 200, 10)
Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_val shape: (1960,)
coord_input_val shape: (1960, 2)
lidar_input_val shape: (1960, 20, 200, 10)
y_train shape: (11194,)
coord_input shape: (11194, 2)
lidar_input shape: (11194, 20, 200, 10)
Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y shape: (9638,)
coord_input shape: (9638, 2)
lidar_input shape: (9638, 20, 200, 10)


/home/matheus/src/RayWise/src/_load_dataset.py:101: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


In [11]:
(
    x_s008_lidar_train,
    x_s008_lidar_val,
    x_s008_coord_train,
    x_s008_coord_val,
    y_s008_train,
    y_val,
) = train_test_split(
    s008_lidar_input,
    s008_coord_input,
    s008_y_train,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

(
    x_s009_lidar_test,
    x_s009_lidar_val,
    x_s009_coord_test,
    x_s009_coord_val,
    y_s009_test,
    y_s009_val,
) = train_test_split(
    s009_lidar_input,
    s009_coord_input,
    s009_y,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

x_lidar_train = x_s008_lidar_train
x_coord_train = x_s008_coord_train
y_train = y_s008_train

x_lidar_val = np.concatenate((x_s008_lidar_val, x_s009_lidar_val), axis=0)
x_coord_val = np.concatenate((x_s008_coord_val, x_s009_coord_val), axis=0)
y_val = np.concatenate((y_val, y_s009_val), axis=0)

x_lidar_test = x_s009_lidar_test
x_coord_test = x_s009_coord_test
y_test = y_s009_test

## 4. Hyperparameters

In [12]:
kparams = KParams.default()
kparams.learning_rate = 7e-5

# kparams = KParams(
#     activation_choices={
#         "relu": tf.keras.activations.relu,
#         "silu": tf.keras.activations.silu, # Swish
#         "tanh": tf.keras.activations.tanh,
#         # "none": None,
#     },
#     regularizer_choices={
#         "l2": tf.keras.regularizers.L2(1e-2),
#         "none": None,
#     },
#     optimizer_choices={
#         "lion": tf.keras.optimizers.Lion(beta_1=0.9, beta_2=0.99),
#     },
#     learning_rate=7e-5,
# )

I0000 00:00:1752751989.602527   67479 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10174 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070, pci bus id: 0000:c3:00.0, compute capability: 8.9


## 5. Model Definition

In [13]:
def build_model(trial: optuna.Trial, kparams: dict, show_summary: bool = True) -> tf.keras.Model:
    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————— Initializer ———————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=TRAIN_SEED,
    )

    # ———————————————————————————————————— GNN ——————————————————————————————————— #
    A = build_knn_adjacency(rows=20, cols=200, k=trial.suggest_int("knn_k", 4, 16, step=4))

    x_graph, a_graph = GraphMasking()([combined, A])

    num_layers = trial.suggest_int("num_gnn_layers", 1, 4)
    for i in range(num_layers):
        # GNN layer with dropout
        x = build_cheb(
            trial=trial,
            kparams=kparams,
            x=x_graph if i == 0 else x,  # Use x_graph only for the first layer
            a_graph=a_graph,
            name_prefix=f"cheb_{i}",
            # Units
            units_range=(128, 1024),
            units_step=128,
            # K
            K_range=(2, 10),
            K_step=1,
            # Dropout
            dropout_rate_range=(0.0, 0.5),
            dropout_rate_step=0.1,
            # Other parameters
            kernel_initializer=initializer,
        )

    # Global pooling layer to reduce the graph to a fixed-size vector
    x = GlobalAvgPool(name="global_avg_pool")(x)

    # ———————————————————————————— Extra dense layers ———————————————————————————— #
    num_dense_layers = trial.suggest_int("num_dense_layers", 0, 3)

    for i in range(num_dense_layers):
        # Dense layer with dropout
        x = build_dnn(
            trial=trial,
            kparams=kparams,
            x=x,
            name_prefix=f"dense_{i}",
            # Units
            units_range=(250, 600),
            units_step=50,
            # Dropout
            dropout_rate_range=(0.0, 0.4),
            dropout_rate_step=0.2,
            # Other parameters
            kernel_initializer=initializer,
        )

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    model.compile(
        optimizer=kparams.get_optimizer(trial),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=False,  # Disable XLA JIT compilation
    )

    return model

## 6. Objective Function

In [14]:
def objective(
    trial: optuna.Trial,
    backup_dir: str,
    model_dir: str,
    fig_dir: str,
    tensorboard_dir: str,
    logs_dir: str,
    history_dir: str,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        X (List[np.ndarray]): List of input arrays.
        y (List[np.ndarray]): List of label arrays.
        backup_dir (str): Path to store backup files.
        model_dir (str): Path to store full models.
        fig_dir (str): Path to store plots.
        tensorboard_dir (str): Path to store TensorBoard logs.
        logs_dir (str): Path to store logs.
        history_dir (str): Path to store training history.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """
    print(f"Running trial {trial.number}...")
    (clear_session(), gc.collect())  # Redundancy cleanup

    # ——————————————————————————————————— Setup —————————————————————————————————— #
    global x_lidar_train
    global x_coord_train
    global y_train
    global x_lidar_val
    global x_coord_val
    global y_val
    global x_lidar_test
    global x_coord_test
    global y_test
    global s009_lidar_input
    global s009_coord_input
    global s009_y

    # Set the random seed for reproducibility
    np.random.seed(TRAIN_SEED)
    tf.random.set_seed(TRAIN_SEED)

    # ———————————————————————————————————————————————————————————————————————————— #

    try:
        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Data Preprocessing                              #
        # ———————————————————————————————————————————————————————————————————————————— #
        coord_scaler = StandardScaler()

        coord_scaler.fit(x_coord_train)
        x_coord_train = coord_scaler.transform(x_coord_train)
        x_coord_val = coord_scaler.transform(x_coord_val)
        x_coord_test = coord_scaler.transform(x_coord_test)
        s009_coord_input = coord_scaler.transform(s009_coord_input)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                        Model Construction and Training                       #
        # ———————————————————————————————————————————————————————————————————————————— #
        model = build_model(trial=trial, kparams=kparams, show_summary=False)

        batch_size = 64
        history = model.fit(
            x=[x_lidar_train, x_coord_train],
            y=y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=get_callbacks_study(
                trial=trial,
                tensorboard_logs=tensorboard_dir,
                monitor="val_loss",
            ),
            verbose=2,
        )

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss_values = punish_model(
            target=history.history["val_loss"],
            model=model,
            type=size_penalizer,
            flops_penalty_factor=1e-10,
            params_penalty_factor=1e-9,
            direction=DIRECTION,
        )
        loss = min(loss_values) if DIRECTION == "minimize" else max(loss_values)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Save Trial Results                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ——————————————————————————— Model characteristics —————————————————————————— #
        set_user_attr_model_stats(
            trial=trial,
            model=model,
            bits_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
            batch_size=batch_size,
            n_trials=1000,
            verbose=True,
        )

        # ————————————————————————————— Evaluate on s009 ————————————————————————————— #
        test_loss, test_acc = model.evaluate(
            [x_lidar_test, x_coord_test], y_test, batch_size=batch_size, verbose=0
        )

        trial.set_user_attr("test_accuracy_s009", float(test_acc))
        trial.set_user_attr("test_loss_s009", float(test_loss))

        # Now evaluate on the full s009 dataset for comparison purposes
        test_loss_full, test_acc_full = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=batch_size, verbose=0
        )
        trial.set_user_attr("test_accuracy_s009_full", float(test_acc_full))
        trial.set_user_attr("test_loss_s009_full", float(test_loss_full))

        # ——————————————————————————————— Save history ——————————————————————————————— #
        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        # Create a DataFrame with all history data
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
        }

        # Add accuracy metrics if available
        if "accuracy" in history.history:
            history_data["train_accuracy"] = history.history["accuracy"]
        if "val_accuracy" in history.history:
            history_data["val_accuracy"] = history.history["val_accuracy"]

        # Convert to DataFrame and save as CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        # ————————————————————————— Finish the current trial ————————————————————————— #
        if len(loss_values) > 1:  # Termination Judgement Report
            report_cross_validation_scores(trial, scores=loss_values)

        return loss  # Value to minimize or maximize

    except optuna.exceptions.TrialPruned:
        raise  # simply propagate pruning
    except tf.errors.ResourceExhaustedError as oom_err:
        print(f"\n❌ Trial {trial.number} hit OOM (Resource Exhausted)\n")
        with open(os.path.join(logs_dir, f"oom_trials.log"), "a") as f:
            f.write(f"OOM error during trial {trial.number}:\n{traceback.format_exc()}\n\n")
        return float("inf") if DIRECTION == "minimize" else float("-inf")
    except Exception as e:
        with open(os.path.join(logs_dir, f"error_trial_{trial.number}.log"), "w") as f:
            f.write(f"An error occurred during trial:\n{e}\n{traceback.format_exc()}\n\n")
        raise  # Re-raise the exception to propagate it
    finally:
        # Clean up resources
        for v in [
            "model",
            "history",
            "history_df",
        ]:
            if v in globals() and globals()[v] is not None:
                del globals()[v]
        (plt.cla(), plt.clf(), plt.close("all"))

## Main

In [15]:
try:
    # ———————————————————————————————— Study Setup ——————————————————————————————— #
    (
        study_dir,
        args_dir,
        fig_dir,
        backup_dir,
        history_dir,
        model_dir,
        logs_dir,
        tensorboard_dir,
    ) = init_study_dirs(RUN_DIR)

    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=f"sqlite:///{study_dir}/optuna_study.db",
        pruner=optuna.pruners.HyperbandPruner(),
        sampler=(optunahub.load_module(package="samplers/auto_sampler")).AutoSampler(),
        load_if_exists=True,
        direction=DIRECTION,
    )

    study.optimize(
        lambda trial: objective(
            trial,
            backup_dir=backup_dir,
            model_dir=model_dir,
            fig_dir=fig_dir,
            logs_dir=logs_dir,
            tensorboard_dir=tensorboard_dir,
            history_dir=history_dir,
            epochs=EPOCHS,
            size_penalizer=None,
        ),
        n_trials=get_remaining_trials(study, NUM_TRIALS),
        callbacks=[
            ImprovementStagnation(variance_threshold=1e-8),
            StopIfKeepBeingPruned(threshold=30),
        ],
        catch=(ValueError, RuntimeError),
        gc_after_trial=True,
    )

    # ——————————————————————— Processing the Study Results ——————————————————————— #
    top_trials = get_top_trials(
        study,
        top_k=TOP_K,
        rank_key=RANK_KEY,  # Use "value" for the study value
        order=ORDER,
    )

    cleanup_paths = [
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
        (history_dir, "trial_{trial_id}.csv"),
        (tensorboard_dir, "trial_{trial_id}"),
    ]

    rename_paths = [
        (model_dir, ".keras"),
        (fig_dir, ".png"),
        (history_dir, ".csv"),
    ]

    extra_attrs = [
        "best_train_accuracy",
        "best_val_accuracy",
        "test_accuracy_s009",
        "test_accuracy_s009_full",
        "test_loss_s009",
        "test_loss_s009_full",
    ]

    save_top_k_trials(
        top_trials,
        args_dir=args_dir,
        study=study,
        extra_attrs=extra_attrs,
    )
    cleanup_non_top_trials(
        {t.number for t in study.trials},  # All trials
        {t.number for t in top_trials},  # Top trials ids
        cleanup_paths,
    )
    rename_top_k_files(top_trials, rename_paths)

    # —————————————————————————— Generate Study Analysis ————————————————————————— #
    (clear(), analyze_study(study, table_dir=os.path.join(study_dir, "analysis")))

except Exception as e:
    print(f"\n An error occurred: {e}\n")
    traceback.print_exc()

    with open(os.path.join(logs_dir, "training_error.log"), "a") as f:
        f.write(f"An error occurred during training:\n{e}\n{traceback.format_exc()}\n\n")
finally:
    # Write success flag for the auto restart script
    Path("/tmp/success.flag").write_text("SUCCESS")

    # Clean up directories
    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)

[I 2025-07-17 08:33:11,976] Using an existing study with name 'optuna_study' instead of creating a new one.


Running trial 2...
Spektral's GCNConv uses a sparse-dense matmul under the hood.
XLA's GPU JIT compiler does not support that op.
This may cause issues with the GNN layers.
Disable all auto-JIT clustering and auto-JIT compilation.
Call:
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=-1'
tf.config.optimizer.set_jit(False)
'jit_compile=False'  # pass into model.compile()
Epoch 1/100


2025-07-17 08:33:27.187830: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:359] gpu_async_0 cuMemAllocAsync failed to allocate 655360000 bytes: RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
 Reported by CUDA: Free memory/Total memory: 86048768/12461604864
2025-07-17 08:33:27.187872: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:364] Stats: Limit:                     10668277760
InUse:                     12116238233
MaxInUse:                  12116238233
NumAllocs:                         315
MaxAllocSize:                917504000
Reserved:                            0
PeakReserved:                        0
LargestFreeBlock:                    0

2025-07-17 08:33:27.187887: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:68] Histogram of current allocation: (allocation_size_in_bytes, nb_allocation_of_that_sizes), ...;
2025-07-17 08:33:27.187892: E external/local_xla/xla/stream_e


❌ Trial 2 hit OOM (Resource Exhausted)



/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/numpy/_core/_methods.py:185: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/optuna/terminator/improvement/evaluator.py:175: RuntimeWarning: invalid value encountered in subtract
  standarized_top_n_values = (top_n_values - top_n_values_mean) / top_n_values_std
[W 2025-07-17 08:33:27,670] The optimization of kernel_params failed: 
linalg.cholesky: The factorization could not be completed because the input is not positive-definite (the leading minor of order 1 is not positive-definite).
The default initial kernel params will be used instead.


Running trial 3...
Epoch 1/100


2025-07-17 08:33:40.493125: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: INVALID_ARGUMENT: Cannot use GPU when output.shape[1] * nnz(a) > 2^31
	 [[{{function_node __inference_one_step_on_data_13552}}{{node functional_1/cheb_1_1/SparseTensorDenseMatMul/SparseTensorDenseMatMul}}]]
[W 2025-07-17 08:33:40,674] Trial 3 failed with parameters: {'knn_k': 12, 'num_gnn_layers': 4, 'cheb_0_units': 640, 'cheb_0_K': 6, 'cheb_0_dropout': 0.0, 'cheb_0_act': 'relu', 'cheb_1_units': 640, 'cheb_1_K': 4, 'cheb_1_dropout': 0.4, 'cheb_1_act': 'elu', 'cheb_2_units': 256, 'cheb_2_K': 9, 'cheb_2_dropout': 0.1, 'cheb_2_act': 'sigmoid', 'cheb_3_units': 128, 'cheb_3_K': 3, 'cheb_3_dropout': 0.5, 'cheb_3_act': 'relu', 'num_dense_layers': 0, 'optimizer': 'sgd'} because of the following error: InvalidArgumentError().
Traceback (most recent call last):
  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/optuna/study/_optimize.py", line 197, i


 An error occurred: Graph execution error:

Detected at node functional_1/cheb_1_1/SparseTensorDenseMatMul/SparseTensorDenseMatMul defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/asyncio/base_events.py", line 1936, in _

Traceback (most recent call last):
  File "/tmp/ipykernel_67479/895900572.py", line 23, in <module>
    study.optimize(
  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/optuna/study/study.py", line 475, in optimize
    _optimize(
  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/optuna/study/_optimize.py", line 63, in _optimize
    _optimize_sequential(
  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/optuna/study/_optimize.py", line 160, in _optimize_sequential
    frozen_trial = _run_trial(study, func, catch)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/optuna/study/_optimize.py", line 248, in _run_trial
    raise func_err
  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/t